# Day 2: Tabular Methods -- Monte Carlo, TD Learning, SARSA & Q-Learning

**Reinforcement Learning: From Theory to Practice**

---

## Overview

Today we move from model-based (Dynamic Programming) to **model-free** methods:

1. **Monte Carlo** prediction and control
2. **Temporal-Difference (TD)** prediction
3. **SARSA** (on-policy TD control)
4. **Q-Learning** (off-policy TD control)
5. **Exploration strategies**: epsilon-greedy, softmax (Boltzmann)

We implement everything from scratch and compare learning curves on simple environments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 0. Simple Environments

We create two lightweight environments so we don't need external dependencies for the tabular methods.

In [ ]:
class CliffWalking:
    """
    The Cliff Walking environment (Sutton & Barto, Example 6.6).
    
    4x12 grid. Start at bottom-left (3,0). Goal at bottom-right (3,11).
    The 'cliff' occupies cells (3,1) through (3,10).
    Stepping on the cliff => reward -100, reset to start.
    Every other step => reward -1. Reaching goal => reward 0, episode ends.
    """
    
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)
        self.goal = (3, 11)
        self.cliff = {(3, c) for c in range(1, 11)}
        self.actions = [0, 1, 2, 3]  # up, right, down, left
        self.action_deltas = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.state = None
    
    def reset(self):
        self.state = self.start
        return self._to_idx(self.state)
    
    def step(self, action):
        r, c = self.state
        dr, dc = self.action_deltas[action]
        nr, nc = r + dr, c + dc
        
        # Boundary check
        nr = max(0, min(self.rows - 1, nr))
        nc = max(0, min(self.cols - 1, nc))
        
        if (nr, nc) in self.cliff:
            self.state = self.start
            return self._to_idx(self.state), -100, False, {}
        
        self.state = (nr, nc)
        
        if self.state == self.goal:
            return self._to_idx(self.state), 0, True, {}
        
        return self._to_idx(self.state), -1, False, {}
    
    def _to_idx(self, rc):
        return rc[0] * self.cols + rc[1]
    
    def _to_rc(self, s):
        return (s // self.cols, s % self.cols)


class WindyGridworld:
    """
    Windy Gridworld (Sutton & Barto, Example 6.5).
    
    7x10 grid. Wind pushes the agent upward in certain columns.
    Start: (3,0), Goal: (3,7).
    """
    
    def __init__(self):
        self.rows = 7
        self.cols = 10
        self.start = (3, 0)
        self.goal = (3, 7)
        self.wind = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]  # upward wind per column
        self.actions = [0, 1, 2, 3]  # up, right, down, left
        self.action_deltas = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.state = None
    
    def reset(self):
        self.state = self.start
        return self._to_idx(self.state)
    
    def step(self, action):
        r, c = self.state
        dr, dc = self.action_deltas[action]
        nr = r + dr - self.wind[c]  # wind pushes up (negative row direction)
        nc = c + dc
        
        nr = max(0, min(self.rows - 1, nr))
        nc = max(0, min(self.cols - 1, nc))
        
        self.state = (nr, nc)
        done = (self.state == self.goal)
        return self._to_idx(self.state), -1, done, {}
    
    def _to_idx(self, rc):
        return rc[0] * self.cols + rc[1]
    
    def _to_rc(self, s):
        return (s // self.cols, s % self.cols)


print('Environments created: CliffWalking (4x12), WindyGridworld (7x10)')

## 1. Exploration Strategies

Before implementing the learning algorithms, we define our exploration policies.

In [ ]:
def epsilon_greedy(Q, state, n_actions, epsilon):
    """
    Epsilon-greedy action selection.
    With probability epsilon, choose a random action.
    Otherwise, choose the greedy action.
    """
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)
    else:
        # Break ties randomly
        q_vals = Q[state]
        max_q = np.max(q_vals)
        best_actions = np.where(q_vals == max_q)[0]
        return np.random.choice(best_actions)


def softmax_action(Q, state, n_actions, temperature=1.0):
    """
    Boltzmann (softmax) action selection.
    Higher temperature => more exploration.
    Lower temperature => more exploitation.
    """
    q_vals = Q[state]
    # Numerical stability: subtract max
    q_shifted = q_vals - np.max(q_vals)
    exp_q = np.exp(q_shifted / max(temperature, 1e-8))
    probs = exp_q / np.sum(exp_q)
    return np.random.choice(n_actions, p=probs)


# Demonstrate the strategies
Q_demo = np.array([1.0, 3.0, 2.0, 0.5])

temps = [0.1, 0.5, 1.0, 5.0]
fig, axes = plt.subplots(1, len(temps), figsize=(16, 3))

for ax, temp in zip(axes, temps):
    counts = np.zeros(4)
    for _ in range(10000):
        q_shifted = Q_demo - np.max(Q_demo)
        exp_q = np.exp(q_shifted / temp)
        probs = exp_q / np.sum(exp_q)
        a = np.random.choice(4, p=probs)
        counts[a] += 1
    ax.bar(range(4), counts / 10000, color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
    ax.set_title(f'Temp = {temp}')
    ax.set_xlabel('Action')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)

plt.suptitle(f'Softmax Exploration (Q = {list(Q_demo)})', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

## 2. Monte Carlo Prediction

Monte Carlo methods estimate $V^\pi(s)$ by averaging returns from complete episodes.

$$V(s) \leftarrow \text{average}\big(G_t \mid S_t = s\big)$$

We use **first-visit MC**: only the first occurrence of each state in an episode contributes.

In [ ]:
def generate_episode(env, Q, n_actions, epsilon, max_steps=200):
    """Generate one episode following epsilon-greedy policy."""
    episode = []
    state = env.reset()
    for _ in range(max_steps):
        action = epsilon_greedy(Q, state, n_actions, epsilon)
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if done:
            break
    return episode


def mc_prediction(env, n_episodes=5000, gamma=1.0, epsilon=0.1):
    """
    First-visit Monte Carlo prediction.
    
    Returns
    -------
    V : estimated state-value function
    """
    Q = np.zeros((env.n_states, env.n_actions))
    V = np.zeros(env.n_states)
    returns_sum = np.zeros(env.n_states)
    returns_count = np.zeros(env.n_states)
    
    for ep in range(n_episodes):
        episode = generate_episode(env, Q, env.n_actions, epsilon)
        
        G = 0.0
        visited = set()
        
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            if state not in visited:  # first-visit
                visited.add(state)
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]
    
    return V


env_cliff = CliffWalking()
V_mc = mc_prediction(env_cliff, n_episodes=10000, gamma=1.0, epsilon=0.1)

# Visualize
grid_v = V_mc.reshape(env_cliff.rows, env_cliff.cols)
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(grid_v, cmap='RdYlGn', interpolation='nearest')
for r in range(env_cliff.rows):
    for c in range(env_cliff.cols):
        ax.text(c, r, f'{grid_v[r,c]:.0f}', ha='center', va='center', fontsize=8)
ax.set_title('MC Prediction -- Cliff Walking (V under epsilon-greedy)', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 3. Monte Carlo Control (with Exploring Starts)

MC control uses Q-values and the epsilon-greedy policy to learn the optimal policy.

In [ ]:
def mc_control(env, n_episodes=10000, gamma=1.0, epsilon=0.1,
               epsilon_decay=0.9999, epsilon_min=0.01):
    """
    First-visit MC control with epsilon-greedy policy.
    
    Returns
    -------
    Q : learned Q-function
    episode_rewards : total reward per episode
    """
    Q = np.zeros((env.n_states, env.n_actions))
    returns_sum = np.zeros((env.n_states, env.n_actions))
    returns_count = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    
    for ep in range(n_episodes):
        episode = generate_episode(env, Q, env.n_actions, epsilon)
        episode_rewards.append(sum(r for _, _, r in episode))
        
        G = 0.0
        visited = set()
        
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward
            
            sa = (state, action)
            if sa not in visited:
                visited.add(sa)
                returns_sum[state, action] += G
                returns_count[state, action] += 1
                Q[state, action] = returns_sum[state, action] / returns_count[state, action]
        
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    return Q, episode_rewards


Q_mc, rewards_mc = mc_control(env_cliff, n_episodes=10000)
print(f'Final avg reward (last 100 eps): {np.mean(rewards_mc[-100:]):.1f}')

## 4. TD(0) Prediction

TD learning updates after every step using the **bootstrap** estimate:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[ R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \right]$$

The quantity $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the **TD error**.

In [ ]:
def td0_prediction(env, n_episodes=5000, alpha=0.1, gamma=1.0, epsilon=0.1,
                   max_steps=200):
    """
    TD(0) prediction following an epsilon-greedy policy.
    """
    V = np.zeros(env.n_states)
    Q = np.zeros((env.n_states, env.n_actions))  # for policy
    
    for ep in range(n_episodes):
        state = env.reset()
        for _ in range(max_steps):
            action = epsilon_greedy(Q, state, env.n_actions, epsilon)
            next_state, reward, done, _ = env.step(action)
            
            # TD update
            td_target = reward + gamma * V[next_state] * (1 - done)
            td_error = td_target - V[state]
            V[state] += alpha * td_error
            
            state = next_state
            if done:
                break
    
    return V


V_td = td0_prediction(env_cliff, n_episodes=10000, alpha=0.1)

# Compare MC vs TD
fig, axes = plt.subplots(1, 2, figsize=(18, 4))

for ax, V, title in zip(axes, [V_mc, V_td], ['Monte Carlo', 'TD(0)']):
    grid_v = V.reshape(env_cliff.rows, env_cliff.cols)
    im = ax.imshow(grid_v, cmap='RdYlGn', interpolation='nearest')
    for r in range(env_cliff.rows):
        for c in range(env_cliff.cols):
            ax.text(c, r, f'{grid_v[r,c]:.0f}', ha='center', va='center', fontsize=8)
    ax.set_title(f'{title} Value Estimate', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 5. SARSA (On-Policy TD Control)

SARSA updates Q-values using the action **actually taken** by the policy:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]$$

The name comes from the quintuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$.

In [ ]:
def sarsa(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1,
          max_steps=500):
    """
    SARSA: on-policy TD control.
    
    Returns
    -------
    Q : learned Q-function
    episode_rewards : list of total rewards per episode
    """
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q, state, env.n_actions, epsilon)
        total_reward = 0
        
        for step in range(max_steps):
            next_state, reward, done, _ = env.step(action)
            next_action = epsilon_greedy(Q, next_state, env.n_actions, epsilon)
            total_reward += reward
            
            # SARSA update
            td_target = reward + gamma * Q[next_state, next_action] * (1 - done)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state = next_state
            action = next_action
            
            if done:
                break
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards


Q_sarsa, rewards_sarsa = sarsa(env_cliff, n_episodes=500, alpha=0.5, epsilon=0.1)
print(f'SARSA -- Final avg reward (last 50 eps): {np.mean(rewards_sarsa[-50:]):.1f}')

## 6. Q-Learning (Off-Policy TD Control)

Q-Learning directly approximates $Q^*$ regardless of the policy being followed:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1,
               max_steps=500):
    """
    Q-Learning: off-policy TD control.
    
    Returns
    -------
    Q : learned Q-function
    episode_rewards : list of total rewards per episode
    """
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            action = epsilon_greedy(Q, state, env.n_actions, epsilon)
            next_state, reward, done, _ = env.step(action)
            total_reward += reward
            
            # Q-Learning update (off-policy: use max over next actions)
            td_target = reward + gamma * np.max(Q[next_state]) * (1 - done)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error
            
            state = next_state
            
            if done:
                break
        
        episode_rewards.append(total_reward)
    
    return Q, episode_rewards


Q_ql, rewards_ql = q_learning(env_cliff, n_episodes=500, alpha=0.5, epsilon=0.1)
print(f'Q-Learning -- Final avg reward (last 50 eps): {np.mean(rewards_ql[-50:]):.1f}')

## 7. Comparison: SARSA vs Q-Learning on Cliff Walking

This is the classic comparison from Sutton & Barto. SARSA learns a **safer** path (away from the cliff) because it is on-policy and accounts for its own exploration. Q-Learning learns the **optimal** path (along the cliff edge) but suffers more during training.

In [ ]:
# Run multiple seeds for a fair comparison
n_runs = 30
n_episodes = 500

sarsa_all = np.zeros((n_runs, n_episodes))
ql_all = np.zeros((n_runs, n_episodes))

for run in range(n_runs):
    np.random.seed(run)
    env_s = CliffWalking()
    env_q = CliffWalking()
    _, rewards_s = sarsa(env_s, n_episodes=n_episodes, alpha=0.5, epsilon=0.1)
    _, rewards_q = q_learning(env_q, n_episodes=n_episodes, alpha=0.5, epsilon=0.1)
    sarsa_all[run] = rewards_s
    ql_all[run] = rewards_q

# Smooth with running average
window = 20

def running_mean(x, w):
    return np.convolve(x, np.ones(w)/w, mode='valid')

sarsa_mean = running_mean(sarsa_all.mean(axis=0), window)
ql_mean = running_mean(ql_all.mean(axis=0), window)

plt.figure(figsize=(10, 5))
plt.plot(sarsa_mean, label='SARSA', linewidth=2, color='#2196F3')
plt.plot(ql_mean, label='Q-Learning', linewidth=2, color='#F44336')
plt.xlabel('Episode')
plt.ylabel('Sum of Rewards (smoothed)')
plt.title('SARSA vs Q-Learning on Cliff Walking')
plt.legend(fontsize=12)
plt.ylim(-150, 0)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the learned policies
action_symbols = {0: '\u2191', 1: '\u2192', 2: '\u2193', 3: '\u2190'}

def plot_policy_cliff(Q, env, title):
    fig, ax = plt.subplots(figsize=(14, 4))
    for r in range(env.rows):
        for c in range(env.cols):
            s = env._to_idx((r, c))
            if (r, c) in env.cliff:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                            color='red', alpha=0.3))
                ax.text(c, r, 'C', ha='center', va='center', fontsize=10, color='red')
            elif (r, c) == env.goal:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                            color='gold', alpha=0.4))
                ax.text(c, r, 'G', ha='center', va='center', fontsize=14, fontweight='bold')
            elif (r, c) == env.start:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=14, fontweight='bold',
                        color='blue')
            else:
                best_a = np.argmax(Q[s])
                ax.text(c, r, action_symbols[best_a], ha='center', va='center', fontsize=16)
    
    ax.set_xlim(-0.5, env.cols - 0.5)
    ax.set_ylim(env.rows - 0.5, -0.5)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.grid(True, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

plot_policy_cliff(Q_sarsa, env_cliff, 'SARSA Policy (safe path)')
plot_policy_cliff(Q_ql, env_cliff, 'Q-Learning Policy (optimal path)')

## 8. Windy Gridworld -- SARSA Learning Curve

In [ ]:
env_windy = WindyGridworld()

Q_windy, rewards_windy = sarsa(env_windy, n_episodes=200, alpha=0.5,
                                gamma=1.0, epsilon=0.1, max_steps=1000)

# Plot time steps per episode
# (In windy gridworld, fewer steps = better)
steps_per_episode = [-r for r in rewards_windy]  # reward is -1 per step

plt.figure(figsize=(10, 4))
plt.plot(steps_per_episode, linewidth=1, alpha=0.5, color='gray')
plt.plot(running_mean(np.array(steps_per_episode, dtype=float), 10),
         linewidth=2, color='#2196F3', label='10-episode average')
plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('SARSA on Windy Gridworld')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal path takes ~15 steps. Final avg: {np.mean(steps_per_episode[-20:]):.0f} steps')

## 9. Effect of Learning Rate and Epsilon

Let us study how hyperparameters affect Q-Learning performance.

In [ ]:
# Effect of alpha (learning rate)
alphas = [0.01, 0.1, 0.3, 0.5, 0.9]
fig, ax = plt.subplots(figsize=(10, 5))

for alpha in alphas:
    np.random.seed(42)
    env_test = CliffWalking()
    _, rewards = q_learning(env_test, n_episodes=500, alpha=alpha, epsilon=0.1)
    smoothed = running_mean(np.array(rewards, dtype=float), 30)
    ax.plot(smoothed, label=f'alpha={alpha}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('Q-Learning: Effect of Learning Rate')
ax.legend()
ax.set_ylim(-200, 0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Effect of epsilon
epsilons = [0.01, 0.05, 0.1, 0.3, 0.5]
fig, ax = plt.subplots(figsize=(10, 5))

for eps in epsilons:
    np.random.seed(42)
    env_test = CliffWalking()
    _, rewards = q_learning(env_test, n_episodes=500, alpha=0.5, epsilon=eps)
    smoothed = running_mean(np.array(rewards, dtype=float), 30)
    ax.plot(smoothed, label=f'eps={eps}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('Q-Learning: Effect of Epsilon')
ax.legend()
ax.set_ylim(-200, 0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Method | Update | On/Off Policy | Requires Full Episodes? |
|--------|--------|---------------|------------------------|
| **MC** | $V(s) \leftarrow V(s) + \alpha[G_t - V(s)]$ | Either | Yes |
| **TD(0)** | $V(s) \leftarrow V(s) + \alpha[r + \gamma V(s') - V(s)]$ | On-policy | No |
| **SARSA** | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q(s',a') - Q(s,a)]$ | On-policy | No |
| **Q-Learning** | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$ | Off-policy | No |

### Key Takeaways

- **Monte Carlo**: Unbiased but high variance; needs complete episodes.
- **TD methods**: Biased (bootstrap) but lower variance; can learn online.
- **SARSA vs Q-Learning**: On-policy (SARSA) learns safe paths; off-policy (Q-Learning) learns optimal paths.
- **Exploration**: epsilon-greedy is simple; softmax (Boltzmann) provides smoother trade-off.

**Next:** Day 3 -- Deep Q-Networks (scaling to continuous state spaces with neural networks)